Runnable Premitive
1. RunnableSequence

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence , RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch

In [3]:
prompt1 = PromptTemplate(
    template="Write a joke about {topic}",
    input_variables=["topic"],
)

model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()
# chain = RunnableSequence(prompt1, model, parser)
# results = chain.invoke(
#     {"topic": "programming"}
# )
# print(results)

prompt2 = PromptTemplate(
    template = "Explain the following joke: {text}",
    input_variables=["text"],
)
chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)
results = chain.invoke(
    {"topic": "programming"}
)
print(results)

This joke plays on the double meaning of the word "bugs." In programming, a "bug" refers to an error or flaw in the code that causes a program to malfunction. The joke suggests that programmers prefer dark mode on their computer screens because the light in light mode attracts literal bugs (insects), which could potentially cause distractions or disruptions while they are trying to work. Additionally, dark mode is often preferred by programmers for its reduced eye strain and improved readability.


Runnable Parrallel

In [5]:
prompt1 = PromptTemplate(
    template="Generate a Tweet about {topic}",
    input_variables=["topic"],
)   
prompt2 = PromptTemplate(
    template="Generate a Linkedin post about {topic}",
    input_variables=["topic"],
)  
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()
parrallel_chain = RunnableParallel({
    "tweet" : RunnableSequence(prompt1, model, parser),
    "linkedin" : RunnableSequence(prompt2, model, parser) 
})

results = parrallel_chain.invoke(
    {"topic": "AI"} 
)
print(results)

{'tweet': '"AI is revolutionizing industries from healthcare to finance, bringing about incredible advancements and efficiencies. The possibilities are truly endless! #AI #technology"', 'linkedin': 'Exciting news in the world of AI! As technology continues to advance, artificial intelligence is revolutionizing industries and changing the way we work. From predictive analytics to personalized recommendations, AI is transforming the way businesses operate. Stay ahead of the curve and learn more about how AI can benefit your organization. #ArtificialIntelligence #AI #TechInnovation #FutureOfWork'}


RunnablePassthrough

In [9]:
prompt1 = PromptTemplate(
    template="Write a joke about {topic}",
    input_variables=["topic"],
)

model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()

prompt2 = PromptTemplate(
    template = "Explain the following joke: {text}",
    input_variables=["text"],
)


chain1 = RunnableSequence(prompt1, model, parser)

parrallel_chain = RunnableParallel({
    "joke": RunnablePassthrough(),
    "explanation": RunnableSequence(prompt2, model, parser)
})
chain2 = RunnableSequence(chain1, parrallel_chain)

results = chain2.invoke(
    {"topic": "programming"}
)
print(results)

{'joke': 'Why do programmers prefer dark mode?\n\nBecause the light attracts bugs!', 'explanation': 'This joke plays on the double meaning of the word "bugs." In programming, a bug is an error or flaw in the code that causes unexpected behavior. By saying that programmers prefer dark mode because the light attracts bugs, it implies that the bugs are literal insects that are attracted to the light. It\'s a play on words that combines programming humor with a common saying about bugs being attracted to light.'}


RunnableLambda

In [12]:
def word_count(text: str) -> int:
    return len(text.split())

runnable_work_counter = RunnableLambda(word_count)
results = runnable_work_counter.invoke(
    "This is a sample text to count the number of words."
)
print(results)

11


In [15]:
prompt1 = PromptTemplate(
    template="Write a joke about {topic}",
    input_variables=["topic"],
)

model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()

prompt2 = PromptTemplate(
    template = "Explain the following joke: {text}",
    input_variables=["text"],
)


chain1 = RunnableSequence(prompt1, model, parser)

# parrallel_chain = RunnableParallel({
#     "joke": RunnablePassthrough(),
#     "word count": RunnableLambda(word_count)
# })
parrallel_chain = RunnableParallel({
    "joke": RunnablePassthrough(),
    "word count": RunnableLambda(lambda x: len(x.split()))
})
chain2 = RunnableSequence(chain1, parrallel_chain)

results = chain2.invoke(
    {"topic": "programming"}
)
print(results)

{'joke': "Why did the programmer quit his job? Because he didn't get arrays!", 'word count': 12}


In [23]:
prompt1 = PromptTemplate(
    template="Write a detailed report about {topic}",
    input_variables=["topic"],
)

model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
parser = StrOutputParser()

prompt2 = PromptTemplate(
    template = "Summarize the following text: {text}",
    input_variables=["text"],
)


report_generation_chain = RunnableSequence(prompt1, model, parser)



parrallel_chain = RunnableBranch(
    
    (lambda x: len(x.split()) >= 100, RunnableSequence(prompt2, model, parser)),
    RunnablePassthrough()
 
)

chain2 = RunnableSequence(report_generation_chain, parrallel_chain)

results = chain2.invoke(
    {"topic": "programming"}
)
print(results)

Programming is the process of writing instructions for a computer to follow to perform a specific task. It involves using different programming languages to write code that is then translated into machine-readable instructions. Programmers need to have a deep understanding of logic, problem-solving, and computer science principles. The process includes defining the problem, designing a solution, writing code, testing, debugging, and deploying the program. Programming is a challenging skill that is essential in modern technology and offers opportunities to create innovative solutions.
